# Data Pipeline For all 14 Datasets

In [5]:

import pandas as pd
import re
import os
import copy

# STEP 0: LOAD ALL 14 DATASETS
PARQUET = "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/raw_data"

hpa_rna         = pd.read_parquet(f"{PARQUET}/1_4_hpa_rna_celline.parquet")
depmap_expr     = pd.read_parquet(f"{PARQUET}/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.parquet")
geo_expr        = pd.read_parquet(f"{PARQUET}/3_GEOexpression.parquet")
proteomics      = pd.read_parquet(f"{PARQUET}/4_Harmonized_MS_CCLE_Gygi_subsetted.parquet")
fusions         = pd.read_parquet(f"{PARQUET}/5_OmicsFusionFilteredSupplementary.parquet")
mutations       = pd.read_parquet(f"{PARQUET}/6_OmicsSomaticMutationsProfile.parquet")
cellosaurus     = pd.read_parquet(f"{PARQUET}/7_cellosaurus.parquet")
depmap_profiles = pd.read_parquet(f"{PARQUET}/8_DepMap_OmicsProfiles.parquet")
sample_info     = pd.read_parquet(f"{PARQUET}/9_DepMap_sample_info.parquet")
geo_info        = pd.read_parquet(f"{PARQUET}/10_GEOInfo.parquet")
hpa_desc        = pd.read_parquet(f"{PARQUET}/11_hpa_rna_celline_description.parquet")
metabolomics    = pd.read_parquet(f"{PARQUET}/12_CCLE_metabolomics_20190502.parquet")
mirna           = pd.read_parquet(f"{PARQUET}/13_CCLE_miRNA_20181103.parquet")
signatures      = pd.read_parquet(f"{PARQUET}/14_OmicsGlobalSignatures.parquet")

tables = {
    "hpa_rna": hpa_rna,
    "depmap_expr": depmap_expr,
    "geo_expr": geo_expr,
    "proteomics": proteomics,
    "fusions": fusions,
    "mutations": mutations,
    "cellosaurus": cellosaurus,
    "depmap_profiles": depmap_profiles,
    "sample_info": sample_info,
    "geo_info": geo_info,
    "hpa_desc": hpa_desc,
    "metabolomics": metabolomics,
    "mirna": mirna,
    "signatures": signatures,
}

# Keep an untouched copy for before/after comparisons
tables_raw = copy.deepcopy(tables)

## 1. HPA_RNA

In [6]:
def clean_hpa_rna(df):
    """
    Clean the HPA RNA cell line table (hpa_rna).

    Steps:
        1. Lowercase all column names
        2. Apply explicit old->new column name mapping
        3. Lowercase all string values
        4. Strip + collapse extra whitespace in string values
        5. In 'gene' column, strip Ensembl version suffix
           (e.g. 'ensg00000000003.15' -> 'ensg00000000003')

    Parameters:
        df (pd.DataFrame): raw hpa_rna table

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).lower() for c in df.columns]

    # --- Step 2: explicit column rename mapping ---
    rename_map = {
        "gene": "gene",
        "gene name": "gene name",
        "cell line": "cell line",
        "tpm": "tpm",
        "ptpm": "ptpm",
        "ntpm": "ntpm"
    }
    df = df.rename(columns=rename_map)

    # --- Step 3 & 4: lowercase values + strip/collapse whitespace ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()                          # leading/trailing whitespace
            .str.replace(r"\s+", " ", regex=True) # collapse internal whitespace
            .str.lower()                          # lowercase
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 5: strip Ensembl version suffix in 'gene' column ---
    # 'ensg00000000003.15' -> 'ensg00000000003'
    if "gene" in df.columns:
        ens_pattern = r"(ens[gtp]\d+)\.\d+"
        df["gene"] = df["gene"].astype(str).str.replace(
            ens_pattern, r"\1", regex=True, flags=re.IGNORECASE
        )
        df["gene"] = df["gene"].replace("nan", pd.NA)

    return df

In [7]:
hpa_rna_clean = clean_hpa_rna(hpa_rna)
hpa_rna_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/hpa_rna_clean.parquet", index=False)

## 2. Depmap_Expr

In [8]:
def clean_depmap_expr(df):
    """
    Clean the DepMap expression table (depmap_expr).

    Structure: rows = PR- profile IDs (index), columns = gene headers
    like 'TSPAN6 (ENSG00000000003)'.

    Steps:
        1. Lowercase column names
        2. Reduce gene headers to bare Ensembl ID:
           'tspan6 (ensg00000000003)' -> 'ensg00000000003'
           'ensg00000288714'          -> 'ensg00000288714' (no brackets, unchanged)
        3. Lowercase string values (index + any object columns)
        4. Strip/collapse extra whitespace
        5. Strip Ensembl version suffix from gene IDs
           ('ensg00000000003.15' -> 'ensg00000000003')

    Parameters:
        df (pd.DataFrame): raw depmap_expr table

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    ens_core    = r"ens[gtp]\d+"          # a bare Ensembl ID
    ens_version = r"(ens[gtp]\d+)\.\d+"   # Ensembl ID + version suffix

    def clean_gene_header(col):
        c = str(col).lower().strip()
        c = re.sub(r"\s+", " ", c)                       # collapse whitespace
        # Step 2: extract the Ensembl ID (works with or without brackets)
        m = re.search(ens_core, c, flags=re.IGNORECASE)
        if m:
            ens = m.group(0).lower()
            # Step 5: strip version suffix if present
            ens = re.sub(ens_version, r"\1", ens, flags=re.IGNORECASE)
            return ens
        return c  # no Ensembl ID found -> keep cleaned original

    # --- Steps 1, 2, 5 on column headers ---
    df.columns = [clean_gene_header(c) for c in df.columns]

    # --- Steps 3 & 4 on the index (PR- IDs) ---
    df.index = (
        df.index.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.lower()
    )
    df.index.name = (df.index.name or "index")

    # --- Steps 3 & 4 on any object/string columns (expression cols are numeric) ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [9]:
depmap_expr_clean = clean_depmap_expr(depmap_expr)
depmap_expr_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/depmap_expr_clean.parquet", index=False)

## 3. Geo_Expr

In [10]:
def clean_geo_expr(df):
    """
    Clean the GEO expression table (geo_expr).

    Structure: rows = genes, columns = ['gene', 'GSM101610', 'GSM101615', ...]

    Steps:
        1. Lowercase column names + all string values
        2. Strip/collapse extra whitespace
        3. In 'gene' column, strip Ensembl version suffix
           ('ensg00000000003.15' -> 'ensg00000000003',
            'ensg00000000005.0'  -> 'ensg00000000005')

    Parameters:
        df (pd.DataFrame): raw geo_expr table

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 1 & 2: lowercase values + strip/collapse whitespace (string cols only) ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 3: strip Ensembl version suffix in 'gene' column ---
    if "gene" in df.columns:
        ens_pattern = r"(ens[gtp]\d+)\.\d+"
        df["gene"] = df["gene"].astype(str).str.replace(
            ens_pattern, r"\1", regex=True, flags=re.IGNORECASE
        )
        df["gene"] = df["gene"].replace("nan", pd.NA)

    return df

In [11]:
geo_expr_clean = clean_geo_expr(geo_expr)
geo_expr_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/geo_expr_clean.parquet", index=False)

## 4. Proteomics

In [12]:
def clean_proteomics(df):
    """
    Clean the proteomics table (proteomics).

    Structure: rows = samples (ach- IDs in 'unnamed: 0'),
    columns = protein headers like 'a0av96 (rbm47)' = uniprot_id (gene_symbol).
    Sometimes the gene symbol is absent: 'a0av96'.

    Steps:
        1. Lowercase column names + all string values
        2. Rename 'unnamed: 0' -> 'depmap_id'
        3. Reduce protein headers to the bare uniprot id, and build a
           separate mapping table {uniprot_id, gene_symbol, original_header}
           so no information is lost.

    Returns:
        (df_clean, protein_map):
            df_clean (pd.DataFrame): matrix with uniprot-id column headers
            protein_map (pd.DataFrame): uniprot_id | gene_symbol | original_header
    """
    df = df.copy()

    # --- Step 1: lowercase column names + strip ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Step 2: rename id column ---
    df = df.rename(columns={"unnamed: 0": "depmap_id"})

    # --- Step 1 (values): lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 3: parse 'uniprot (gene)' headers, build mapping, rename cols ---
    # matches: 'a0av96 (rbm47)'  -> id='a0av96', gene='rbm47'
    #          'a0av96'          -> id='a0av96', gene=None
    pattern = re.compile(r"^\s*([a-z0-9\-]+)\s*(?:\(([^)]*)\))?\s*$", re.IGNORECASE)

    id_cols = ["depmap_id"]  # columns to leave untouched
    map_rows = []
    new_names = {}

    for col in df.columns:
        if col in id_cols:
            continue
        m = pattern.match(str(col))
        if m:
            uniprot_id = m.group(1).strip().lower()
            gene       = (m.group(2).strip().lower() if m.group(2) else pd.NA)
        else:
            uniprot_id = str(col).strip().lower()
            gene       = pd.NA

        new_names[col] = uniprot_id
        map_rows.append({
            "uniprot_id": uniprot_id,
            "gene_symbol": gene,
            "original_header": col,
        })

    df = df.rename(columns=new_names)
    protein_map = pd.DataFrame(map_rows)

    return df, protein_map

In [13]:
proteomics_clean, protein_map = clean_proteomics(proteomics)

proteomics_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/proteomics_clean.parquet", index=False)
protein_map.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/protein_map.parquet", index=False)

## 5. Fusions

In [14]:
def clean_fusions(df):
    """
    Clean the fusions table (fusions).

    Steps:
        1. Lowercase column names
        2. Lowercase all string values
        3. Rename 'unnamed: 0' -> 'fusion_index'
        4. Split 'gene1(ens id)'  -> 'gene1', 'gene1_ens_id'
        5. Split 'gene2(ens id)'  -> 'gene2', 'gene2_ens_id'
        6. Strip/collapse whitespace
        7. Strip Ensembl version suffix (.21/.0/.00) from the ens id parts
        8. Empty ()        -> ens id = NA (empty)
           Placeholder (.) -> kept as-is ('.', etc.)

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Step 3: rename id column ---
    df = df.rename(columns={"unnamed: 0": "fusion_index"})

    # --- Steps 2 & 6: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- helper to split 'gene (ens id)' into (gene, ens_id) ---
    # captures: group1 = text before '(', group2 = text inside '()'
    split_pat = re.compile(r"^(.*?)\s*\(([^)]*)\)\s*$")
    ens_version = re.compile(r"(ens[gtp]\d+)\.\d+", re.IGNORECASE)

    def split_gene_ens(val):
        if pd.isna(val):
            return (pd.NA, pd.NA)
        s = str(val).strip()
        m = split_pat.match(s)
        if m:
            gene = m.group(1).strip()
            ens  = m.group(2).strip()
            # Step 7: strip version suffix only on real ensembl IDs
            ens = ens_version.sub(r"\1", ens)
            # Step 8: empty () -> NA; placeholder like '.' kept as-is
            if ens == "":
                ens = pd.NA
            gene = gene if gene != "" else pd.NA
            return (gene, ens)
        # no parentheses found -> whole value is the gene, no ens id
        return (s if s != "" else pd.NA, pd.NA)

    # --- Steps 4, 5, 7, 8 ---
    for src, gene_col, ens_col in [
        ("gene1(ens id)", "gene1", "gene1_ens_id"),
        ("gene2(ens id)", "gene2", "gene2_ens_id"),
    ]:
        if src in df.columns:
            parsed = df[src].apply(split_gene_ens)
            df[gene_col] = parsed.apply(lambda x: x[0])
            df[ens_col]  = parsed.apply(lambda x: x[1])
            df = df.drop(columns=[src])

    return df

In [15]:
fusions_clean = clean_fusions(fusions)
fusions_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/fusions_clean.parquet", index=False)

## 6. Mutations

In [16]:
def clean_mutations(df):
    """
    Clean the mutations table (mutations).

    Structure: ~1.07M rows x 70 cols. Sample key = 'profileid' (PR-).
    Ensembl IDs live in 'ensemblgeneid' (and 'ensemblfeatureid').

    Steps:
        1. Lowercase all column names
        2. Strip/collapse whitespace in string values
        3. Strip Ensembl version suffix (.5/.00/.0) from ensembl ID columns
        4. Lowercase all string values

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 4: whitespace clean + lowercase on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 3: strip Ensembl version suffix on ensembl ID columns only ---
    ens_pattern = r"(ens[gtp]\d+)\.\d+"
    ens_cols = [c for c in ["ensemblgeneid", "ensemblfeatureid"] if c in df.columns]
    for col in ens_cols:
        df[col] = df[col].astype(str).str.replace(
            ens_pattern, r"\1", regex=True, flags=re.IGNORECASE
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [17]:
mutations_clean = clean_mutations(mutations)
mutations_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/mutations_clean.parquet", index=False)

## 7. Cellosaurus

In [18]:
def clean_cellosaurus(df):
    """
    Clean the cellosaurus table (cellosaurus).

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4/5. Rename:
            'identifier (cell line name)' -> 'cellosaurus_cell_line_name'
            'accession (cvcl_xxxx)'       -> 'cellosaurus_accession'

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Steps 4 & 5: rename key columns ---
    rename_map = {
        "identifier (cell line name)": "cellosaurus_cell_line_name",
        "accession (cvcl_xxxx)": "cellosaurus_accession",
    }
    df = df.rename(columns=rename_map)

    return df

In [19]:
cellosaurus_clean = clean_cellosaurus(cellosaurus)
cellosaurus_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/cellosaurus_clean.parquet", index=False)

## 8. Depmap_Profiles

In [20]:
def clean_depmap_profiles(df):
    """
    Clean the depmap_profiles table (depmap_profiles).

    This is the bridge table linking PR- profile IDs to ACH- model IDs.

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [21]:
depmap_profiles_clean = clean_depmap_profiles(depmap_profiles)
depmap_profiles_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/depmap_profiles_clean.parquet", index=False)

## 9. sample_info

In [22]:
def clean_sample_info(df):
    """
    Clean the sample_info table (sample_info).

    Central cell-line dimension table. Key columns include depmap_id (ACH-),
    cell_line_name, stripped_cell_line_name, ccle_name, cosmicid, rrid, etc.

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [23]:
sample_info_clean = clean_sample_info(sample_info)
sample_info_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/sample_info_clean.parquet", index=False)

## 10. Geo Info

In [24]:
def clean_geo_info(df):
    """
    Clean the geo_info table (geo_info).

    GEO sample metadata. Key join column is geo_accession (GSM IDs).
    NOTE: this table had known leading/trailing whitespace on the cell line
    field causing silent join failures — the whitespace strip here fixes that.

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [25]:
geo_info_clean = clean_geo_info(geo_info)
geo_info_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/geo_info_clean.parquet", index=False)

## 11. HPA Desc

In [26]:
def clean_hpa_desc(df):
    """
    Clean the hpa_desc table (hpa_desc).

    HPA cell line metadata. Key columns: 'cell line' (join to sample_info /
    cellosaurus), 'cellosaurus id', disease info.

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [27]:
hpa_desc_clean = clean_hpa_desc(hpa_desc)
hpa_desc_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/hpa_desc_clean.parquet", index=False)

## 12. Metabolomics

In [28]:
def clean_metabolomics(df):
    """
    Clean the metabolomics table (metabolomics).

    Wide CCLE metabolomics. Keys: 'ccle_id' and 'depmap_id' (ACH-).
    Remaining ~225 columns are metabolite abundances (numeric).

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [29]:
metabolomics_clean = clean_metabolomics(metabolomics)
metabolomics_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/metabolomics_clean.parquet", index=False)

## 13. Mirna

In [30]:
def clean_mirna(df):
    """
    Clean the mirna table (mirna).

    Structure: 'name' (miRNA id, e.g. hsa-mir-21), 'description',
    then ~954 cell-line columns CCLE-style (e.g. 'dms53_lung').

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4. In 'name', strip trailing decimal suffix (.0 / .00 / .000)

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    # --- Step 4: strip trailing decimal suffix in 'name' ---
    if "name" in df.columns:
        df["name"] = df["name"].astype(str).str.replace(r"\.0+$", "", regex=True)
        df["name"] = df["name"].replace("nan", pd.NA)

    return df

In [31]:
mirna_clean = clean_mirna(mirna)
mirna_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/mirna_clean.parquet", index=False)


## 14. Signatures

In [32]:
def clean_signatures(df):
    """
    Clean the signatures table (signatures).

    Pre-computed global omics signature scores. Connects via modelid (ACH-)
    and sequencingid (PR-).

    Steps:
        1. Lowercase all column names
        2. Lowercase all string values
        3. Strip/collapse whitespace in string values
        4. Rename 'unnamed: 0' -> 'signature_index'

    Returns:
        pd.DataFrame: cleaned copy
    """
    df = df.copy()

    # --- Step 1: lowercase column names ---
    df.columns = [str(c).strip().lower() for c in df.columns]

    # --- Step 4: rename id column ---
    df = df.rename(columns={"unnamed: 0": "signature_index"})

    # --- Steps 2 & 3: lowercase + whitespace clean on string cols ---
    str_cols = df.select_dtypes(include=["object", "str"]).columns
    for col in str_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower()
        )
        df[col] = df[col].replace("nan", pd.NA)

    return df

In [33]:
signatures_clean = clean_signatures(signatures)
signatures_clean.to_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/signatures_clean.parquet", index=False)

## Saving Clean Paraquets and Table Summary in Excel File

In [34]:
import pandas as pd

# ---------------------------------------------------------------
# LOAD ALL CLEAN TABLES
# ---------------------------------------------------------------
table_clean = {
    "hpa_rna_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/hpa_rna_clean.parquet"),
    "depmap_expr_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/depmap_expr_clean.parquet"),
    "geo_expr_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/geo_expr_clean.parquet"),
    "proteomics_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/proteomics_clean.parquet"),
    "protein_map_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/protein_map.parquet"),
    "fusions_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/fusions_clean.parquet"),
    "mutations_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/mutations_clean.parquet"),
    "cellosaurus_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/cellosaurus_clean.parquet"),
    "depmap_profiles_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/depmap_profiles_clean.parquet"),
    "sample_info_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/sample_info_clean.parquet"),
    "geo_info_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/geo_info_clean.parquet"),
    "hpa_desc_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/hpa_desc_clean.parquet"),
    "metabolomics_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/metabolomics_clean.parquet"),
    "mirna_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/mirna_clean.parquet"),
    "signatures_clean": pd.read_parquet("/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/parquet/data_clean/signatures_clean.parquet")
}

# ---------------------------------------------------------------
# BUILD COLUMN-LEVEL SUMMARY
# ---------------------------------------------------------------
def build_table_summary(df, n_samples=3):
    """
    Per-column summary:
    column | dtype | sample_values | missing_% | unique_values
           | primary_key | duplicates
    """
    rows = []
    n_rows = len(df)

    for col in df.columns:
        s = df[col]

        n_missing = s.isna().sum()
        n_present = n_rows - n_missing
        n_unique = s.nunique(dropna=True)
        n_duplicates = n_present - n_unique
        missing_pct = round((n_missing / n_rows) * 100, 2) if n_rows else 0

        is_pk = (n_missing == 0) and (n_duplicates == 0) and (n_rows > 0)

        sample = s.dropna().astype(str).head(n_samples).tolist()

        rows.append({
            "column": col,
            "dtype": str(s.dtype),
            "sample_values": ", ".join(sample),
            "missing_%": missing_pct,
            "unique_values": n_unique,
            "primary_key": is_pk,
            "duplicates": n_duplicates
        })

    return pd.DataFrame(rows)


# ---------------------------------------------------------------
# EXPORT TO EXCEL (ONE SHEET PER TABLE)
# ---------------------------------------------------------------
def export_clean_report(tables: dict, out_path, n_samples=3):

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for name, df in tables.items():

            summary = build_table_summary(df, n_samples=n_samples)
            sheet_name = name[:31]  # Excel limit

            header = pd.DataFrame({
                "info": [
                    f"TABLE: {name}",
                    f"shape: {df.shape[0]} rows x {df.shape[1]} cols"
                ]
            })

            header.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
                header=False,
                startrow=0
            )

            summary.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
                startrow=3
            )

    print(f"Written {len(tables)} sheets to: {out_path}")


# ---------------------------------------------------------------
# RUN REPORT
# ---------------------------------------------------------------
OUT = "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/clean_data_report.xlsx"

export_clean_report(table_clean, OUT, n_samples=3)

Written 15 sheets to: /Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/data/clean_data_report.xlsx


## Checking Data Overlap in every Table

In [36]:
"""
=====================================================================
VALUE-BASED JOIN DISCOVERY
Find connections between tables by overlapping ACTUAL VALUES,
not column names.
=====================================================================
"""

import pandas as pd
from itertools import combinations


# ---------------------------------------------------------------
# STEP 1: Build a value-set "fingerprint" for each candidate key column
# ---------------------------------------------------------------
# We only fingerprint columns that look like ID/key columns — pure value
# columns (expression floats, etc.) would be noise. Heuristic: low-to-mid
# cardinality string columns, plus anything matching known ID patterns.

import re

ID_HINTS = re.compile(
    r"(id|name|accession|gsm|cvcl|ach|pr-|profile|model|gene|cell|symbol|ensembl|ccle|rrid|cosmic)",
    re.IGNORECASE,
)

def get_key_columns(df, max_unique_frac=0.99, min_unique=2, sample_n=200_000):
    """
    Pick columns worth fingerprinting:
      - string/object dtype OR name matches an ID hint
      - more than `min_unique` distinct values
    For very large tables, sample rows to keep it fast.
    """
    if len(df) > sample_n:
        df = df.sample(sample_n, random_state=0)

    cols = []
    for col in df.columns:
        name_hit = bool(ID_HINTS.search(str(col)))
        is_str = df[col].dtype == object or str(df[col].dtype) == "string"
        if not (name_hit or is_str):
            continue
        nun = df[col].nunique(dropna=True)
        if nun >= min_unique:
            cols.append(col)
    return cols


def build_fingerprints(tables: dict, sample_n=200_000):
    """
    For each table, build {column: set(values)} for candidate key columns.
    Values are stringified + lowercased for fair comparison.
    """
    fingerprints = {}
    for name, df in tables.items():
        key_cols = get_key_columns(df, sample_n=sample_n)
        df_use = df.sample(sample_n, random_state=0) if len(df) > sample_n else df

        col_sets = {}
        for col in key_cols:
            vals = (
                df_use[col].dropna().astype(str).str.strip().str.lower().unique()
            )
            col_sets[col] = set(vals)
        fingerprints[name] = col_sets
        print(f"{name:18s}: fingerprinted {len(col_sets)} candidate key columns")
    return fingerprints


# ---------------------------------------------------------------
# STEP 2: Compare every column-pair ACROSS tables for value overlap
# ---------------------------------------------------------------

def find_connections(fingerprints: dict, min_overlap=5, min_pct=5.0):
    """
    Compare each column in each table against each column in every other
    table. Report pairs whose values overlap.

    Parameters:
        min_overlap (int): minimum number of shared values to report
        min_pct (float): minimum overlap % (of the smaller set) to report

    Returns:
        pd.DataFrame of candidate joins, sorted by overlap strength.
    """
    rows = []
    table_names = list(fingerprints.keys())

    for t1, t2 in combinations(table_names, 2):
        for col1, set1 in fingerprints[t1].items():
            for col2, set2 in fingerprints[t2].items():
                if not set1 or not set2:
                    continue
                inter = set1 & set2
                n = len(inter)
                if n < min_overlap:
                    continue

                smaller = min(len(set1), len(set2))
                pct_smaller = n / smaller * 100
                if pct_smaller < min_pct:
                    continue

                rows.append({
                    "table_1": t1,
                    "column_1": col1,
                    "table_2": t2,
                    "column_2": col2,
                    "n_overlap": n,
                    "n_distinct_1": len(set1),
                    "n_distinct_2": len(set2),
                    "pct_of_smaller": round(pct_smaller, 1),
                    "example_shared": list(inter)[:3]
                })

    result = pd.DataFrame(rows)
    if not result.empty:
        result = result.sort_values(
            ["pct_of_smaller", "n_overlap"], ascending=False
        ).reset_index(drop=True)
    return result


# ---------------------------------------------------------------
# RUN
# ---------------------------------------------------------------
print("=" * 60)
print("STEP 1: Building value fingerprints")
print("=" * 60)
fingerprints = build_fingerprints(table_clean, sample_n=200_000)

print("\n" + "=" * 60)
print("STEP 2: Finding value-based connections")
print("=" * 60)
connections = find_connections(fingerprints, min_overlap=5, min_pct=5.0)

print(f"\nFound {len(connections)} candidate join connections.\n")
print(connections.to_string(index=False))

STEP 1: Building value fingerprints
hpa_rna_clean     : fingerprinted 3 candidate key columns
depmap_expr_clean : fingerprinted 0 candidate key columns
geo_expr_clean    : fingerprinted 3268 candidate key columns
proteomics_clean  : fingerprinted 7 candidate key columns
protein_map_clean : fingerprinted 3 candidate key columns
fusions_clean     : fingerprinted 21 candidate key columns
mutations_clean   : fingerprinted 47 candidate key columns
cellosaurus_clean : fingerprinted 17 candidate key columns
depmap_profiles_clean: fingerprinted 5 candidate key columns
sample_info_clean : fingerprinted 29 candidate key columns
geo_info_clean    : fingerprinted 19 candidate key columns
hpa_desc_clean    : fingerprinted 7 candidate key columns
metabolomics_clean: fingerprinted 18 candidate key columns
mirna_clean       : fingerprinted 242 candidate key columns
signatures_clean  : fingerprinted 7 candidate key columns

STEP 2: Finding value-based connections

Found 118 candidate join connections.
